# Lead AI · Infrastructure industrielle et orchestration

**Objectif :** Infrastructure as Code avec Terraform, réseau AWS et Airflow.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Infrastructure as Code : Terraform et réseau AWS

Créer des serveurs, des bases et des réseaux en cliquant dans la console AWS ne se rejoue pas, ne se relit pas en revue de code, et dérive avec le temps. L'**Infrastructure as Code** décrit l'infrastructure dans des fichiers texte, versionnés dans Git. **Terraform** (ou son fork open source **OpenTofu**) est l'outil le plus répandu, et fonctionne avec tous les clouds.

## Le cycle de Terraform

```python
# main.tf
provider "aws" { region = "eu-west-3" }

resource "aws_s3_bucket" "modeles" {
  bucket = "fraude-modeles-prod"
}

resource "aws_instance" "api" {
  ami           = "ami-0abcdef1234567890"
  instance_type = var.type_instance
  subnet_id     = aws_subnet.prive_a.id       # une référence : Terraform en déduit l'ordre de création
  tags          = { Name = "fraude-api" }
}

variable "type_instance" { default = "t3.small" }
output "ip_api" { value = aws_instance.api.private_ip }
```

1. `terraform init` : télécharge les fournisseurs (*providers*).
2. `terraform plan` : compare la configuration à l'**état** actuel et affiche ce qui va changer : `+` créer, `~` modifier sur place, `-` détruire, `-/+` détruire et recréer (quand un attribut ne peut pas changer sur place, comme l'image d'une instance).
3. `terraform apply` : applique le plan.

Comme Kubernetes, Terraform est **déclaratif** : on décrit le résultat voulu, l'outil calcule les étapes.

## L'état, et pourquoi il doit être distant

Terraform garde dans un fichier d'**état** (*state*) la correspondance entre la configuration et les ressources réelles. En équipe, ce fichier doit être **partagé et verrouillé** : stocké dans un bucket S3, avec un verrou (table DynamoDB, ou le verrou natif de S3) pour que deux personnes n'appliquent pas en même temps. Il contient des données sensibles : il ne va jamais dans Git.

## Le réseau AWS en un schéma

- Un **VPC** : ton réseau privé, avec une plage d'adresses (par exemple `10.0.0.0/16`, 65 536 adresses).
- Des **sous-réseaux** dans plusieurs zones de disponibilité : **publics** (avec une route vers une Internet Gateway) pour le load balancer, **privés** pour l'API, les bases et les GPU.
- Une **NAT Gateway** pour que les sous-réseaux privés sortent vers Internet (télécharger des paquets) sans être joignables de l'extérieur.
- Des **groupes de sécurité** (vague 6) sur chaque ressource.

La notation CIDR `10.0.1.0/24` signifie : les 24 premiers bits sont fixés, il reste 8 bits, soit 256 adresses.

## Les modules

Un **module** Terraform regroupe des ressources réutilisables (« un VPC standard », « une API sur ECS »), paramétrées par des variables. Le registre public en propose des milliers, dont le module VPC officiel d'AWS.

> **À retenir** : init, plan, apply. Le plan compare l'état à la configuration (+, ~, -, -/+). État distant et verrouillé, jamais dans Git. VPC, sous-réseaux publics et privés, NAT, groupes de sécurité.

**À faire :**

- [ ] Installer Terraform (ou OpenTofu) et créer un bucket S3 avec terraform apply, puis le détruire avec terraform destroy
- [ ] Configurer un état distant dans S3 avec verrouillage
- [ ] Créer un VPC avec deux sous-réseaux publics et deux privés grâce au module terraform-aws-modules/vpc

### Exercice 1 · Calculer un plan

Écris `planifier(etat, config, immuables=("ami", "instance_type"))` : `etat` et `config` associent un nom de ressource à un dictionnaire d'attributs. Renvoie la liste des actions, triée par nom : `("+", nom)` pour une ressource à créer, `("-", nom)` pour une ressource à détruire, et pour une ressource dont des attributs changent (ajoutés, retirés ou modifiés) : `("-/+", nom, attributs)` si l'un d'eux est immuable, sinon `("~", nom, attributs)`, où `attributs` est la liste triée des attributs qui changent. Une ressource identique n'apparaît pas.

<details><summary>Indice</summary>

Parcours sorted(set(etat) | set(config)). Pour une ressource présente des deux côtés, compare a.get(k) et b.get(k) sur l'union des clés.
</details>

In [ ]:
def planifier(etat, config, immuables=("ami", "instance_type")):
    pass

etat = {
    "aws_s3_bucket.modeles": {"bucket": "fraude-modeles"},
    "aws_instance.api": {"ami": "ami-1", "instance_type": "t3.small", "tags": {"Name": "api"}},
    "aws_instance.batch": {"ami": "ami-1", "instance_type": "t3.large"},
    "aws_security_group.api": {"ingress": [443]},
}
config = {
    "aws_s3_bucket.modeles": {"bucket": "fraude-modeles"},
    "aws_instance.api": {"ami": "ami-1", "instance_type": "t3.small", "tags": {"Name": "api", "equipe": "ml"}},
    "aws_instance.batch": {"ami": "ami-2", "instance_type": "t3.large"},
    "aws_db_instance.suivi": {"engine": "postgres"},
}

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = planifier(etat, config)
attendu = [("+", "aws_db_instance.suivi"), ("~", "aws_instance.api", ["tags"]), ("-/+", "aws_instance.batch", ["ami"]), ("-", "aws_security_group.api")]
assert r == attendu, f"Plan attendu :\n{attendu}\nTu obtiens :\n{r}"
assert planifier(etat, etat) == [], "Sans changement, le plan est vide."
assert planifier({"x": {"a": 1}}, {"x": {"a": 1, "b": 2}}) == [("~", "x", ["b"])], "Un attribut ajouté compte comme un changement."
assert planifier({"x": {"a": 1}}, {"x": {"a": 2}}, immuables=("a",)) == [("-/+", "x", ["a"])], "Respecte le paramètre immuables."
print("Bravo, tout est correct.")

### Exercice 2 · Découper un VPC

Avec le module `ipaddress` de la bibliothèque standard, écris `decouper(cidr_vpc, nombre, prefixe)` qui renvoie les `nombre` premiers sous-réseaux de taille `/prefixe` du VPC, sous forme de chaînes ; `taille(cidr)`, le nombre d'adresses d'un bloc ; et `appartient(ip, cidr)`, True si l'adresse est dans le bloc.

<details><summary>Indice</summary>

ipaddress.ip_network(cidr).subnets(new_prefix=prefixe) génère les sous-réseaux ; prends-en les premiers ; num_addresses donne la taille.
</details>

In [ ]:
import ipaddress

def decouper(cidr_vpc, nombre, prefixe):
    pass

def taille(cidr):
    pass

def appartient(ip, cidr):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert decouper("10.0.0.0/16", 3, 24) == ["10.0.0.0/24", "10.0.1.0/24", "10.0.2.0/24"], "Les trois premiers /24 de 10.0.0.0/16."
assert decouper("10.0.0.0/16", 2, 20) == ["10.0.0.0/20", "10.0.16.0/20"], "Avec un préfixe /20, les blocs font 4 096 adresses."
assert taille("10.0.0.0/16") == 65536 and taille("10.0.1.0/24") == 256, "Un /16 a 65 536 adresses, un /24 en a 256."
assert appartient("10.0.1.42", "10.0.1.0/24") and not appartient("10.0.2.1", "10.0.1.0/24"), "10.0.1.42 est dans 10.0.1.0/24, pas 10.0.2.1."
print("Bravo, tout est correct.")

### Exercice 3 · L'ordre de création

Terraform crée les ressources dans un ordre compatible avec leurs références. Écris `ordre_creation(dependances)` : `dependances` associe chaque ressource à la liste de celles qu'elle référence. Renvoie une liste où chaque ressource vient après ses dépendances ; parmi les ressources prêtes au même moment, prends l'ordre **alphabétique**. S'il y a un cycle, lève une `ValueError`.

<details><summary>Indice</summary>

Algorithme de Kahn : garde pour chaque ressource l'ensemble de ses dépendances restantes. Tant qu'il en reste, prends la plus petite ressource (alphabétiquement) sans dépendance, ajoute-la, et retire-la des autres ensembles.
</details>

In [ ]:
def ordre_creation(dependances):
    pass

dependances = {
    "aws_instance.api": ["aws_subnet.prive", "aws_security_group.api"],
    "aws_subnet.prive": ["aws_vpc.principal"],
    "aws_security_group.api": ["aws_vpc.principal"],
    "aws_vpc.principal": [],
    "aws_lb.public": ["aws_subnet.public", "aws_security_group.lb"],
    "aws_subnet.public": ["aws_vpc.principal"],
    "aws_security_group.lb": ["aws_vpc.principal"],
}

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = ordre_creation(dependances)
attendu = ["aws_vpc.principal", "aws_security_group.api", "aws_security_group.lb", "aws_subnet.prive", "aws_instance.api", "aws_subnet.public", "aws_lb.public"]
assert r == attendu, f"Ordre attendu (à chaque étape, la première ressource prête par ordre alphabétique) :\n{attendu}\nTu obtiens :\n{r}"
assert ordre_creation({"b": ["a"]}) == ["a", "b"], "Une ressource référencée mais non listée doit aussi apparaître."
try:
    ordre_creation({"a": ["b"], "b": ["a"]}); cycle = False
except ValueError:
    cycle = True
assert cycle, "Un cycle doit lever une ValueError."
print("Bravo, tout est correct.")

---
## Orchestration avec Airflow

Un pipeline ML réel enchaîne des étapes : extraire les transactions de la veille, contrôler leur qualité, recalculer les variables, réentraîner, évaluer, publier le modèle si c'est mieux. Chaque nuit, avec des reprises en cas d'échec et des alertes. C'est le travail d'un **orchestrateur**. **Apache Airflow** est le plus utilisé ; Dagster et Prefect sont des alternatives modernes. Tu as vu les bases dans la leçon ETL de la vague 1 ; ici, on passe au niveau production.

## Les notions clés

- **DAG** (*directed acyclic graph*) : le pipeline, un graphe de tâches sans cycle.
- **Tâche** : une étape, créée à partir d'un **opérateur** (Python, Bash, SQL, Kubernetes, Spark…).
- **Planification** : `schedule="@daily"`, ou une expression cron (`"0 2 * * *"` = tous les jours à 2 h).
- **Intervalle de données** : chaque exécution traite une période précise. L'exécution du 15 mars à 2 h traite les données du 14 mars, de 00:00 à 24:00.
- **Reprises** : `retries=3`, `retry_delay=timedelta(minutes=5)` pour les pannes passagères.
- **Rattrapage** (*backfill*, *catchup*) : rejouer automatiquement les périodes manquées.

*Animation interactive « dagrun » : à voir sur la plateforme web.*

```python
from datetime import datetime, timedelta
from airflow.decorators import dag, task

@dag(schedule="@daily", start_date=datetime(2026, 1, 1), catchup=False,
     default_args={"retries": 2, "retry_delay": timedelta(minutes=5)})
def fraude_quotidien():

    @task
    def extraire(data_interval_start=None, data_interval_end=None):
        return requete_sql(debut=data_interval_start, fin=data_interval_end)

    @task
    def controler(chemin):
        verifier_contrat(chemin)        # échoue si les données sont invalides : rien en aval ne tourne

    @task
    def entrainer(chemin): ...

    @task
    def publier_si_meilleur(run_id): ...

    donnees = extraire()
    controler(donnees) >> entrainer(donnees)

fraude_quotidien()
```

## Les règles d'or

- **Idempotence** : relancer une tâche pour la même période doit donner le même résultat, sans doublon (on écrase la partition du jour au lieu d'ajouter des lignes).
- **Des tâches atomiques** : une tâche fait une chose ; si elle échoue, on la relance seule.
- **Pas de gros calcul dans Airflow lui-même** : il orchestre. Le calcul part dans un conteneur, un job Kubernetes ou Spark.
- **Pas de données entre tâches**, seulement des références (un chemin S3, un identifiant de run MLflow).

> **À retenir** : DAG de tâches, planification par intervalles de données, reprises, rattrapage. Des tâches idempotentes et atomiques qui orchestrent le calcul sans le faire.

**À faire :**

- [ ] Lancer Airflow avec Docker Compose (guide officiel « Running Airflow in Docker »)
- [ ] Écrire un DAG quotidien qui extrait des données Immo, contrôle leur qualité et réentraîne un modèle
- [ ] Provoquer un échec, observer les reprises, puis relancer une période passée avec airflow dags backfill

### Exercice 1 · Exécuter un DAG avec reprises

Écris `executer(taches, dependances, retries=1)` : `taches` associe un nom à une fonction sans argument, `dependances` associe un nom à la liste des tâches dont il dépend. Exécute les tâches dans l'ordre (parmi les prêtes, ordre alphabétique). Une tâche qui lève une exception est relancée jusqu'à `retries` fois de plus ; si elle échoue encore, son statut est `"failed"`, et toutes les tâches qui en dépendent, directement ou non, prennent le statut `"upstream_failed"` sans être exécutées. Les autres prennent `"success"`. Renvoie le dictionnaire des statuts.

<details><summary>Indice</summary>

Boucle tant que toutes les tâches n'ont pas de statut. Une tâche est prête quand toutes ses dépendances ont un statut ; si l'une n'est pas success, elle devient upstream_failed sans être exécutée.
</details>

In [ ]:
def executer(taches, dependances, retries=1):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
journal = []
def ok(nom):
    return lambda: journal.append(nom)
def ko():
    journal.append("ko"); raise RuntimeError("panne")
essais = {"n": 0}
def instable():
    essais["n"] += 1; journal.append("instable")
    if essais["n"] < 3: raise ConnectionError("réseau")
T = {"extraire": ok("extraire"), "controler": ko, "entrainer": ok("entrainer"), "rapport": ok("rapport"), "publier": ok("publier")}
D = {"controler": ["extraire"], "entrainer": ["controler"], "publier": ["entrainer"], "rapport": ["extraire"]}
r = executer(T, D, retries=2)
assert r == {"extraire": "success", "controler": "failed", "entrainer": "upstream_failed", "publier": "upstream_failed", "rapport": "success"}, f"Statuts inattendus : {r}"
assert journal.count("ko") == 3, f"Avec retries=2, une tâche en échec est tentée 3 fois ; elle l'a été {journal.count('ko')} fois."
assert "entrainer" not in journal and "publier" not in journal, "Les tâches en aval d'un échec ne doivent pas être exécutées."
assert journal.index("controler" if "controler" in journal else "ko") < journal.index("rapport"), "Parmi les tâches prêtes, l'ordre est alphabétique : controler avant rapport."
journal.clear()
assert executer({"a": instable, "b": ok("b")}, {"b": ["a"]}, retries=2) == {"a": "success", "b": "success"}, "Une tâche qui réussit à la 3e tentative est un succès."
print("Bravo, tout est correct.")

### Exercice 2 · Les intervalles de données

Écris `intervalles(debut, maintenant, catchup=True)` pour un DAG quotidien : `debut` et `maintenant` sont des `datetime`. Chaque exécution couvre un jour complet `[J 00:00, J+1 00:00)`, et ne peut tourner qu'une fois l'intervalle **terminé**. Renvoie la liste des tuples `(debut_intervalle, fin_intervalle)` à exécuter, du plus ancien au plus récent, en commençant au jour de `debut` (à minuit). Avec `catchup=False`, renvoie seulement le dernier intervalle terminé (une liste d'un élément, ou vide).

<details><summary>Indice</summary>

Pars de minuit le jour de debut ; ajoute des intervalles d'un jour tant que leur fin est <= maintenant. Pour catchup=False, res[-1:].
</details>

In [ ]:
from datetime import datetime, timedelta

def intervalles(debut, maintenant, catchup=True):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from datetime import datetime
r = intervalles(datetime(2026, 3, 10, 15, 30), datetime(2026, 3, 13, 2, 0))
assert r == [(datetime(2026, 3, 10), datetime(2026, 3, 11)), (datetime(2026, 3, 11), datetime(2026, 3, 12)), (datetime(2026, 3, 12), datetime(2026, 3, 13))], f"Trois intervalles terminés (10, 11 et 12 mars) ; tu as {r}."
assert intervalles(datetime(2026, 3, 10), datetime(2026, 3, 13, 2, 0), catchup=False) == [(datetime(2026, 3, 12), datetime(2026, 3, 13))], "Sans rattrapage, seul le dernier intervalle terminé est exécuté."
assert intervalles(datetime(2026, 3, 10), datetime(2026, 3, 10, 23, 59)) == [], "L'intervalle du jour n'est pas terminé : rien à exécuter."
assert intervalles(datetime(2026, 3, 10), datetime(2026, 3, 11)) == [(datetime(2026, 3, 10), datetime(2026, 3, 11))], "Un intervalle qui se termine exactement maintenant peut tourner."
print("Bravo, tout est correct.")